# Simulación del Modelo Predictivo (Minuto 15) para Web
Aquí vamos a simular el comportamiento de la futura web. En la web el usuario seleccionará los equipos, los jugadores, los campeones y añadirá las estadísticas de la partida en el minuto 15. Luego, por detrás (backend), extraeremos el histórico de esos jugadores/campeones/equipos para completar las variables necesarias del modelo y ejecutar la predicción de la partida.

In [35]:
import pandas as pd
import os

# Creamos la carpeta si no existe
os.makedirs('../data/experimentos', exist_ok=True)

# Cargamos el archivo de 2026
df_26 = pd.read_csv('../data/filtered/LECdata-2026.csv', low_memory=False)

# Filtramos solo las filas de equipo
df_equipos_26 = df_26[df_26['position'] == 'team'].copy()

# Seleccionamos las columnas base de partida/equipo que usamos antes de calcular win ratios
columnas_base = [
    'gameid', 'playoffs', 'side', 'teamname', 'firstdragon', 
    'golddiffat15', 'xpdiffat15', 'csdiffat15', 
    'killsat15', 'assistsat15', 'deathsat15', 'result'
]

df_equipos_26 = df_equipos_26[columnas_base]

# Guardamos el archivo resultante
df_equipos_26.to_csv('../data/experimentos/LECdata-2026-equipos.csv', index=False)
print("Archivo guardado en data/experimentos/LECdata-2026-equipos.csv")

Archivo guardado en data/experimentos/LECdata-2026-equipos.csv


In [36]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

# 1. CARGA DE MODELOS PREENTRENADOS
log_reg_model = joblib.load('../models/modelo_regresion_logistica.pkl')
scaler = joblib.load('../models/scaler.pkl')
label_encoders = joblib.load('../models/label_encoders.pkl')

# 2. CARGA DE HISTÓRICO PARA BACKEND (Se cargará en memoria en la web)
df_2021 = pd.read_csv('../data/filtered/LECdata-2021.csv', low_memory=False)
df_2022 = pd.read_csv('../data/filtered/LECdata-2022.csv', low_memory=False)
df_2023 = pd.read_csv('../data/filtered/LECdata-2023.csv', low_memory=False)
df_2024 = pd.read_csv('../data/filtered/LECdata-2024.csv', low_memory=False)
df_2025 = pd.read_csv('../data/filtered/LECdata-2025.csv', low_memory=False)

df_historico = pd.concat([df_2021, df_2022, df_2023, df_2024, df_2025], ignore_index=True)
df_historico['teamname'] = df_historico['teamname'].replace({'MAD Lions KOI': 'KOI', 'Movistar KOI': 'KOI'})

# Función para extraer el estado "actual" (último ewm) de win rates en base al historial
def calcular_historial_backend(df):
    # Separamos en jugadores y equipos
    df_jugadores = df[df['position'] != 'team'].copy()
    df_equipos = df[df['position'] == 'team'].copy()
    
    dict_stats = {}
    
    # 2.1 WR de Jugadores
    df_jugadores['player_win_ratio'] = df_jugadores.groupby('playername')['result'].transform(lambda x: x.ewm(span=20, min_periods=1).mean().round(4))
    df_jugadores['player_games'] = df_jugadores.groupby('playername')['result'].transform(lambda x: x.expanding().count())
    last_player = df_jugadores.drop_duplicates(subset=['playername'], keep='last')
    
    dict_stats['players'] = {}
    for _, row in last_player.iterrows():
        wr = 0.5 if row['player_games'] < 20 else row['player_win_ratio']
        dict_stats['players'][row['playername']] = wr

    # 2.2 WR de Campeones
    df_jugadores['champ_win_ratio'] = df_jugadores.groupby('champion')['result'].transform(lambda x: x.ewm(span=20, min_periods=1).mean().round(4))
    df_jugadores['champ_wr_games'] = df_jugadores.groupby('champion')['result'].transform(lambda x: x.expanding().count())
    last_champ = df_jugadores.drop_duplicates(subset=['champion'], keep='last')
    
    dict_stats['champs_wr'] = {}
    for _, row in last_champ.iterrows():
        wr = 0.5 if row['champ_wr_games'] < 20 else row['champ_win_ratio']
        dict_stats['champs_wr'][row['champion']] = wr

    # 2.3 Early Power de Campeones (oro al min 15)
    # Rellenamos nulos por si acaso
    df_jugadores['golddiffat15'] = df_jugadores['golddiffat15'].fillna(0)
    df_jugadores['champ_gold_mean'] = df_jugadores.groupby('champion')['golddiffat15'].transform(lambda x: x.ewm(span=20, min_periods=1).mean().round(4))
    df_jugadores['champ_games'] = df_jugadores.groupby('champion')['golddiffat15'].transform(lambda x: x.expanding().count())
    last_champ_gold = df_jugadores.drop_duplicates(subset=['champion'], keep='last')
    
    dict_stats['champs_early'] = {}
    for _, row in last_champ_gold.iterrows():
        gold_mean = 0 if row['champ_games'] < 20 else row['champ_gold_mean']
        dict_stats['champs_early'][row['champion']] = gold_mean
        
    # 2.4 WR de Equipos
    df_equipos['team_wr_val'] = df_equipos.groupby('teamname')['result'].transform(lambda x: x.ewm(span=20, min_periods=1).mean().round(4))
    df_equipos['team_games'] = df_equipos.groupby('teamname')['result'].transform(lambda x: x.expanding().count())
    last_team = df_equipos.drop_duplicates(subset=['teamname'], keep='last')
    
    dict_stats['teams'] = {}
    for _, row in last_team.iterrows():
        wr = 0.5 if row['team_games'] < 20 else row['team_wr_val']
        dict_stats['teams'][row['teamname']] = wr
        
    return dict_stats

backend_stats = calcular_historial_backend(df_historico)
print("Backend preparado. Históricos calculados ✅")

Backend preparado. Históricos calculados ✅


In [37]:
# 4. LÓGICA DE BACKEND PARA PREPARAR DATOS
def procesar_equipo(datos_equipo, is_blue, form_stats, backend_dict):
    """
    Toma los datos del formulario de un equipo y los cruza con el historial 
    para fabricar las 27 variables que espera el modelo.
    """
    equipo = datos_equipo['teamname']
    
    # Cálculos en min 15 respecto a la perspectiva del equipo actual
    signo = 1 if is_blue else -1
    
    # 1. Variables directas del formulario
    playoffs = datos_equipo['playoffs']
    side = datos_equipo['side']
    firstdragon = 1 if form_stats['first_dragon_team'] == equipo else 0
    golddiffat15 = form_stats['stats_min_15']['gold_diff'] * signo
    xpdiffat15 = form_stats['stats_min_15']['xp_diff'] * signo
    csdiffat15 = form_stats['stats_min_15']['cs_diff'] * signo
    
    if is_blue:
        killsat15 = form_stats['stats_min_15']['kills_azul']
        assistsat15 = form_stats['stats_min_15']['assists_azul']
        deathsat15 = form_stats['stats_min_15']['deaths_azul']
    else:
        killsat15 = form_stats['stats_min_15']['kills_rojo']
        assistsat15 = form_stats['stats_min_15']['assists_rojo']
        deathsat15 = form_stats['stats_min_15']['deaths_rojo']
        
    # Variables de campeones
    champ_top = datos_equipo['jugadores']['top']['campeon']
    champ_jng = datos_equipo['jugadores']['jng']['campeon']
    champ_mid = datos_equipo['jugadores']['mid']['campeon']
    champ_bot = datos_equipo['jugadores']['bot']['campeon']
    champ_sup = datos_equipo['jugadores']['sup']['campeon']
    
    # 2. Variables históricas (cruzar con diccionario del backend)
    team_wr = backend_dict['teams'].get(equipo, 0.5)
    wr_top = backend_dict['players'].get(datos_equipo['jugadores']['top']['nombre'], 0.5)
    wr_jng = backend_dict['players'].get(datos_equipo['jugadores']['jng']['nombre'], 0.5)
    wr_mid = backend_dict['players'].get(datos_equipo['jugadores']['mid']['nombre'], 0.5)
    wr_bot = backend_dict['players'].get(datos_equipo['jugadores']['bot']['nombre'], 0.5)
    wr_sup = backend_dict['players'].get(datos_equipo['jugadores']['sup']['nombre'], 0.5)
    
    wr_champ_top = backend_dict['champs_wr'].get(champ_top, 0.5)
    wr_champ_jng = backend_dict['champs_wr'].get(champ_jng, 0.5)
    wr_champ_mid = backend_dict['champs_wr'].get(champ_mid, 0.5)
    wr_champ_bot = backend_dict['champs_wr'].get(champ_bot, 0.5)
    wr_champ_sup = backend_dict['champs_wr'].get(champ_sup, 0.5)
    
    comp_early_power = (
        backend_dict['champs_early'].get(champ_top, 0) +
        backend_dict['champs_early'].get(champ_jng, 0) +
        backend_dict['champs_early'].get(champ_mid, 0) +
        backend_dict['champs_early'].get(champ_bot, 0) +
        backend_dict['champs_early'].get(champ_sup, 0)
    )
    
    # 3. Creación del Diccionario de características
    return {
        'playoffs': playoffs, 'side': side, 'teamname': equipo, 'team_wr': team_wr,
        'champ_top': champ_top, 'champ_jng': champ_jng, 'champ_mid': champ_mid, 
        'champ_bot': champ_bot, 'champ_sup': champ_sup, 
        'wr_champ_top': wr_champ_top, 'wr_champ_jng': wr_champ_jng, 'wr_champ_mid': wr_champ_mid, 
        'wr_champ_bot': wr_champ_bot, 'wr_champ_sup': wr_champ_sup,
        'firstdragon': firstdragon, 'golddiffat15': golddiffat15, 'xpdiffat15': xpdiffat15, 
        'csdiffat15': csdiffat15, 'killsat15': killsat15, 'assistsat15': assistsat15, 'deathsat15': deathsat15,
        'wr_top': wr_top, 'wr_jng': wr_jng, 'wr_mid': wr_mid, 'wr_bot': wr_bot, 'wr_sup': wr_sup,
        'comp_early_power': comp_early_power
    }

# Columnas categóricas que el modelo necesita que codifiquemos luego
target_cols = ['side', 'teamname', 'champ_top', 'champ_jng', 'champ_mid', 'champ_bot', 'champ_sup']


- ***Experimento Múltiple:***

In [38]:
# Función automatizada para simular partidas masivas por ID de Game
def simular_partida_por_id(game_id, df_datos_completos=df_26):
    """
    Rastrea el game_id en el dataset original del año, construye el formulario simulado
    como si lo metiera el usuario, y ejecuta el pipeline de predicción calculando el promedio. 
    """
    print(f"--- SIMULANDO PARTIDA: {game_id} ---")
    game_data = df_datos_completos[df_datos_completos['gameid'] == game_id]
    
    if game_data.empty:
        print("❌ Error: Game ID no encontrado.")
        return False
    
    blue_data = game_data[game_data['side'] == 'Blue']
    red_data = game_data[game_data['side'] == 'Red']
    
    # Extraemos info base de las filas "team"
    blue_team = blue_data[blue_data['position'] == 'team'].iloc[0]
    red_team = red_data[red_data['position'] == 'team'].iloc[0]
    
    # Extraemos información de los jugadores
    def get_players(side_data):
        pos_map = ['top', 'jng', 'mid', 'bot', 'sup']
        return {
            pos: {
                "nombre": side_data[side_data['position']==pos]['playername'].values[0],
                "campeon": side_data[side_data['position']==pos]['champion'].values[0]
            } for pos in pos_map
        }
        
    first_dragon_team = blue_team['teamname'] if blue_team['firstdragon'] == 1 else (red_team['teamname'] if red_team['firstdragon'] == 1 else "None")
    
    # 1. Simular Formulario Web Estructurado
    form_stats = {
        "first_dragon_team": first_dragon_team,
        "stats_min_15": {
            "kills_azul": blue_team['killsat15'], "kills_rojo": red_team['killsat15'],
            "assists_azul": blue_team['assistsat15'], "assists_rojo": red_team['assistsat15'],
            "deaths_azul": blue_team['deathsat15'], "deaths_rojo": red_team['deathsat15'],
            "gold_diff": blue_team['golddiffat15'],   # La del azul es la absoluta (si es -, va perdiendo)
            "xp_diff": blue_team['xpdiffat15'],
            "cs_diff": blue_team['csdiffat15']
        }
    }
    
    formulario = {
        "equipo_azul": {
            "teamname": blue_team['teamname'],
            "playoffs": blue_team['playoffs'],
            "side": "Blue",
            "jugadores": get_players(blue_data)
        },
        "equipo_rojo": {
            "teamname": red_team['teamname'],
            "playoffs": red_team['playoffs'],
            "side": "Red",
            "jugadores": get_players(red_data)
        }
    }
    
    # 2. Paso por Backend (Armar variables)
    df_input_azul = pd.DataFrame([procesar_equipo(formulario['equipo_azul'], True, form_stats, backend_stats)])
    df_input_rojo = pd.DataFrame([procesar_equipo(formulario['equipo_rojo'], False, form_stats, backend_stats)])
    df_input = pd.concat([df_input_azul, df_input_rojo], ignore_index=True)
    
    # 3. Preprocesado y Codificación
    for col in target_cols:
        if col in label_encoders:
            le = label_encoders[col]
            df_input[col] = df_input[col].map(lambda s: le.transform([str(s)])[0] if str(s) in le.classes_ else -1)
            
    X_predict = scaler.transform(df_input)
    
    # 4. Predicción y Balanceo de Promedio
    probabilidades = log_reg_model.predict_proba(X_predict)
    prob_azul_bruta, prob_rojo_bruta = probabilidades[0][1], probabilidades[1][1]
    
    prob_azul_final = (prob_azul_bruta + (1.0 - prob_rojo_bruta)) / 2.0
    prob_rojo_final = (prob_rojo_bruta + (1.0 - prob_azul_bruta)) / 2.0
    
    # 5. Salida de Resultados
    equipo_azul_nombre = blue_team['teamname']
    equipo_rojo_nombre = red_team['teamname']
    
    print(f"➤ VICTORIA {equipo_azul_nombre} (Azul): {prob_azul_final * 100:.2f}%")
    print(f"➤ VICTORIA {equipo_rojo_nombre} (Rojo): {prob_rojo_final * 100:.2f}%")
    
    ganador_real = equipo_azul_nombre if blue_team['result'] == 1 else equipo_rojo_nombre
    prediccion_correcta = (prob_azul_final > prob_rojo_final and blue_team['result'] == 1) or (prob_rojo_final > prob_azul_final and red_team['result'] == 1)
    
    prediccion = equipo_azul_nombre if prob_azul_final > prob_rojo_final else equipo_rojo_nombre
    print(f"\nPredicción Modelo: {prediccion} 🏆")
    print(f"Resultado Real: {ganador_real}")
    print(f"{'✅ ACIERTO' if prediccion_correcta else '❌ FALLO'}\n")
    
    return prediccion_correcta

# Extraemos 30 partidas aleatorias o las primeras del 2026 para probar la función:
# ids_de_prueba = df_26['gameid'].unique()[:30]
ids_de_prueba = np.random.choice(df_26['gameid'].unique(), size=30, replace=False)

contador_fallos = 0
for idx in ids_de_prueba:
    acierto = simular_partida_por_id(idx)
    if not acierto:
        contador_fallos += 1

print(f"=========================================")
print(f"Total de partidas simuladas: {len(ids_de_prueba)}")
print(f"Total de aciertos: {len(ids_de_prueba) - contador_fallos} ({(len(ids_de_prueba) - contador_fallos) / len(ids_de_prueba) * 100:.2f}%)")
print(f"Total de fallos: {contador_fallos} ({contador_fallos / len(ids_de_prueba) * 100:.2f}%)")
print(f"=========================================")

--- SIMULANDO PARTIDA: LOLTMNT04_156150 ---
➤ VICTORIA GiantX (Azul): 94.44%
➤ VICTORIA Team Heretics (Rojo): 5.56%

Predicción Modelo: GiantX 🏆
Resultado Real: GiantX
✅ ACIERTO

--- SIMULANDO PARTIDA: LOLTMNT04_156685 ---
➤ VICTORIA GiantX (Azul): 68.95%
➤ VICTORIA Shifters (Rojo): 31.05%

Predicción Modelo: GiantX 🏆
Resultado Real: GiantX
✅ ACIERTO

--- SIMULANDO PARTIDA: LOLTMNT04_155130 ---
➤ VICTORIA GiantX (Azul): 71.25%
➤ VICTORIA Team Heretics (Rojo): 28.75%

Predicción Modelo: GiantX 🏆
Resultado Real: GiantX
✅ ACIERTO

--- SIMULANDO PARTIDA: LOLTMNT04_159298 ---
➤ VICTORIA Movistar KOI (Azul): 45.37%
➤ VICTORIA Shifters (Rojo): 54.63%

Predicción Modelo: Shifters 🏆
Resultado Real: Shifters
✅ ACIERTO

--- SIMULANDO PARTIDA: LOLTMNT04_149122 ---
➤ VICTORIA Natus Vincere (Azul): 61.37%
➤ VICTORIA Karmine Corp Blue (Rojo): 38.63%

Predicción Modelo: Natus Vincere 🏆
Resultado Real: Natus Vincere
✅ ACIERTO

--- SIMULANDO PARTIDA: LOLTMNT04_161335 ---
➤ VICTORIA Movistar KOI (Azul): 